# BÀI TẬP 2: "MLP CHALLENGE" — CIFAR-10 BẰNG MẠNG NƠ-RON NHIỀU LỚP
## Thiết kế kiến trúc Deep Residual MLP tuân thủ 100% yêu cầu đề bài

Notebook này được chuẩn hóa nghiêm ngặt theo đúng yêu cầu trong `DE_BAI.md`:
* **Đầu vào:** Làm phẳng ảnh $(32 \times 32 \times 3)$ thành vector **3072 chiều** (không dùng lớp tích chập Conv2d/CNN, không chia patch như MLP-Mixer).
* **Kiến trúc:** Mạng nơ-ron nhiều lớp hoàn toàn tuyến tính (**Deep Residual MLP**) với các khối Linear, BatchNorm1d, GELU và shortcut Residual Connection.
* **Tối ưu hóa & Kỹ thuật nâng cao:**
  * **E1 (Optimizer):** AdamW tối ưu trọng số và hội tụ nhanh, ổn định.
  * **E2 (Khởi tạo):** Kaiming Normal (He) Initialization phù hợp hàm kích hoạt phi tuyến.
  * **E3 (Lịch LR):** Linear Warmup (3 epochs) kết hợp Cosine Annealing Scheduler.
  * **E4 (Regularization):** Weight Decay ($2 \times 10^{-2}$), Dropout (0.25), Data Augmentation (Crop, Flip, Rotation, ColorJitter trên không gian ảnh trước khi flatten) và Early Stopping.
  * **E5 (Residual):** Kết nối tắt $F(x) + x$ giữa các khối Linear giúp dòng gradient lan truyền ổn định ở mạng sâu.
  * **Huấn luyện:** Stratified 3-Fold Cross Validation + Mixed Precision (AMP - tối ưu card RTX 4060) + Ensemble Soft-voting trên tập Test.


In [1]:
import os
import time
import copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.model_selection import StratifiedKFold


## 1. Cấu hình Siêu tham số (Hyperparameters & Setup)
- Thiết lập đường dẫn thư mục `data/` và `submissions/` tương đối chuẩn trong workspace.
- Cố định Seed đảm bảo tính tái lập (Reproducibility).
- Tận dụng GPU NVIDIA GeForce RTX 4060 Laptop nếu có CUDA.


In [2]:
DATA_DIR = Path('data')
OUT_DIR = Path('submissions')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS = 3
EPOCHS = 80
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 2e-2
DROPOUT_RATE = 0.25
PATIENCE = 10

def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"Sử dụng GPU: {torch.cuda.get_device_name(0)}")
        return device
    print("Sử dụng CPU")
    return torch.device('cpu')


## 2. Tiền xử lý dữ liệu & Data Augmentation
- Dữ liệu ảnh $(32 \times 32 \times 3)$ được áp dụng Data Augmentation (Crop, Lật, Xoay nhẹ, Jitter).
- Sau khi augment, dữ liệu **bắt buộc được làm phẳng thành vector 3072 chiều** theo đúng yêu cầu đề bài.


In [3]:
def load_raw_data():
    train = np.load(DATA_DIR / "train.npz")
    test = np.load(DATA_DIR / "test.npz")
    
    # Định dạng (N, C, H, W) trong khoảng [0, 1] để tiện augment
    x = train["images"].transpose(0, 3, 1, 2).astype(np.float32) / 255.0
    y = train["labels"].astype(np.int64)
    x_test = test["images"].transpose(0, 3, 1, 2).astype(np.float32) / 255.0
    
    print(f"Đã nạp Train: {x.shape}, Test: {x_test.shape}")
    return x, y, x_test, test["ids"]


class CIFARMLPDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        self.images = torch.from_numpy(images)
        self.labels = torch.from_numpy(labels) if labels is not None else None
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        # Làm phẳng ảnh thành vector 3072 chiều cho MLP thuần
        img_flat = torch.flatten(img)
        if self.labels is not None:
            return img_flat, self.labels[idx]
        return img_flat


## 3. Kiến trúc Deep Residual MLP (Tuân thủ đề bài E2 & E5)
- **Đặc điểm:** Không dùng bất kỳ lớp tích chập (`Conv2d`), toàn bộ sử dụng các tầng tuyến tính (`nn.Linear`).
- **Khối MLP Residual Block:**
  $$y = \text{GELU}(\text{BN}(\text{Linear}_2(\text{Dropout}(\text{GELU}(\text{BN}(\text{Linear}_1(x)))))) + x)$$
  Cơ chế tắt (residual connection) giải quyết triệt để vấn đề tiêu biến đạo hàm (vanishing gradient) khi tăng chiều sâu mạng.
- **Khởi tạo trọng số (E2):** Áp dụng Kaiming Normal (He) cho toàn bộ các lớp `Linear`.


In [4]:
class MLPResidualBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.25):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.act1 = nn.GELU()
        self.drop = nn.Dropout(dropout)
        
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.act2 = nn.GELU()

    def forward(self, x):
        residual = x
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.act1(out)
        out = self.drop(out)
        out = self.fc2(out)
        out = self.bn2(out)
        return self.act2(out + residual)


class DeepResidualMLP(nn.Module):
    def __init__(self, in_features: int = 3072, hidden_dim: int = 1024, num_classes: int = 10, 
                 num_blocks: int = 5, dropout: float = 0.25):
        super().__init__()
        # Chiếu từ không gian đầu vào 3072 chiều lên không gian ẩn
        self.in_proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU()
        )
        
        # Xếp chồng các khối MLP Residual
        self.blocks = nn.ModuleList([
            MLPResidualBlock(hidden_dim, dropout=dropout) for _ in range(num_blocks)
        ])
        
        # Tầng phân loại đầu ra
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.in_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.classifier(x)


## 4. Hàm Đánh giá (Evaluation: Loss, Accuracy & RMSE)
Đánh giá mô hình trên tập validation với các độ đo:
- **Loss:** Cross Entropy Loss.
- **Accuracy:** Tỉ lệ dự đoán chính xác lớp.
- **RMSE:** Sai số căn quân phương của phân phối xác suất dự đoán so với one-hot nhãn thực.


In [5]:
@torch.no_grad()
def evaluate(model, loader, device, num_classes=10):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    squared_error_sum = 0.0
    criterion = nn.CrossEntropyLoss(reduction="sum")

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(xb)
            loss_sum += criterion(logits, yb).item()

        preds = logits.argmax(1)
        correct += (preds == yb).sum().item()
        total += len(yb)

        probs = torch.softmax(logits, dim=1)
        one_hot = torch.zeros_like(probs).scatter_(1, yb.unsqueeze(1), 1.0)
        squared_error_sum += torch.sum((probs - one_hot) ** 2).item()

    rmse = np.sqrt(squared_error_sum / (total * num_classes))
    return loss_sum / total, correct / total, rmse


## 5. Huấn luyện Stratified K-Fold & Xuất File Nộp Bài
- Sử dụng Stratified 3-Fold Cross Validation.
- Hỗ trợ **Mixed Precision (AMP)** với `torch.amp.GradScaler` giúp tăng tốc độ xử lý trên RTX 4060.
- Kết hợp **Linear Warmup + Cosine Annealing Learning Rate Scheduler**.
- **Early Stopping:** Dừng khi sau `PATIENCE` epoch không cải thiện Val Accuracy.
- **Ensemble:** Trung bình xác suất dự đoán từ 3 fold trên tập test -> xuất file `submissions/submission_mlp.csv`.


In [6]:
def main():
    set_seed(SEED)
    device = get_device()
    x, y, x_test, test_ids = load_raw_data()

    train_transform = T.Compose([
        T.RandomCrop(32, padding=4, padding_mode='reflect'),
        T.RandomHorizontalFlip(),
        T.RandomRotation(15),
        T.ColorJitter(brightness=0.15, contrast=0.15),
    ])

    test_ds = CIFARMLPDataset(x_test, transform=None)
    test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_val_accs = []
    oof_val_rmses = []
    test_preds_folds = []

    print(f"\n================ BẮT ĐẦU HUẤN LUYỆN TRÊN {device.type.upper()} ================")
    use_amp = (device.type == 'cuda')

    for fold, (train_idx, val_idx) in enumerate(skf.split(x, y)):
        print(f"\n>>> FOLD {fold + 1}/{N_SPLITS}")

        train_ds = CIFARMLPDataset(x[train_idx], y[train_idx], transform=train_transform)
        val_ds = CIFARMLPDataset(x[val_idx], y[val_idx], transform=None)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

        model = DeepResidualMLP(hidden_dim=1024, num_blocks=5, dropout=DROPOUT_RATE).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

        warmup_epochs = 3
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
        cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(EPOCHS - warmup_epochs), eta_min=1e-6)
        scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_epochs])

        best_val_acc = 0.0
        best_val_rmse = 0.0
        best_model_weights = None
        no_improve_epochs = 0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            t0 = time.time()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                
                with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                    logits = model(xb)
                    loss = criterion(logits, yb)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            scheduler.step()

            tr_loss, tr_acc, _ = evaluate(model, train_loader, device)
            va_loss, va_acc, va_rmse = evaluate(model, val_loader, device)

            print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
                  f"Val Loss: {va_loss:.4f} Acc: {va_acc:.4f} RMSE: {va_rmse:.4f} | {time.time() - t0:.1f}s")

            if va_acc > best_val_acc:
                best_val_acc = va_acc
                best_val_rmse = va_rmse
                best_model_weights = copy.deepcopy(model.state_dict())
                no_improve_epochs = 0
            else:
                no_improve_epochs += 1

            if no_improve_epochs >= PATIENCE:
                print(f"--> Early Stopping tại Epoch {epoch}.")
                break

        print(f"\n=> Tốt nhất Fold {fold + 1}: Val Acc = {best_val_acc:.4f} | Val RMSE = {best_val_rmse:.4f}")
        oof_val_accs.append(best_val_acc)
        oof_val_rmses.append(best_val_rmse)

        # Dự đoán test fold
        model.load_state_dict(best_model_weights)
        model.eval()
        fold_probs = []
        with torch.no_grad():
            for xb in test_loader:
                xb = xb.to(device)
                with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                    logits = model(xb)
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                fold_probs.append(probs)
        test_preds_folds.append(np.concatenate(fold_probs, axis=0))

    print("\n================ TỔNG KẾT MÔ HÌNH ================")
    print(f"Mean Val Acc: {np.mean(oof_val_accs):.4f} +/- {np.std(oof_val_accs):.4f}")
    print(f"Mean Val RMSE: {np.mean(oof_val_rmses):.4f} +/- {np.std(oof_val_rmses):.4f}")

    avg_test_probs = np.mean(test_preds_folds, axis=0)
    final_preds = np.argmax(avg_test_probs, axis=1)

    out_file = OUT_DIR / "submission_mlp.csv"
    with open(out_file, "w") as f:
        f.write("id,label\n")
        for i, p in zip(test_ids, final_preds):
            f.write(f"{i},{p}\n")
    print(f"\nĐã lưu kết quả tại: {out_file}")
    
    print(f"Tổng số mẫu dự đoán: {len(final_preds)}")
    print(f"Phân bố các nhãn:")
    unique, counts = np.unique(final_preds, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"  Lớp {u}: {c} ảnh")


if __name__ == '__main__':
    main()


Sử dụng GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Đã nạp Train: (50000, 3, 32, 32), Test: (10000, 3, 32, 32)

================ BẮT ĐẦU HUẤN LUYỆN TRÊN CUDA ================

>>> FOLD 1/3
Epoch  1/80 | Train Loss: 18.9571 Acc: 0.2179 | Val Loss: 18.9524 Acc: 0.2420 RMSE: 0.3767 | 34.3s
Epoch  2/80 | Train Loss: 7.0263 Acc: 0.2959 | Val Loss: 7.2555 Acc: 0.3220 RMSE: 0.3256 | 33.3s


d:\4-University Stuff Season 16\KI 1\Mang Noron\DuDoanHinhAnh\ai_env\lib\site-packages\torch\optim\lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch  3/80 | Train Loss: 6.4589 Acc: 0.3070 | Val Loss: 7.2175 Acc: 0.3399 RMSE: 0.3048 | 32.7s
Epoch  4/80 | Train Loss: 3.8475 Acc: 0.3285 | Val Loss: 4.7487 Acc: 0.3674 RMSE: 0.2916 | 35.3s
Epoch  5/80 | Train Loss: 3.3733 Acc: 0.3491 | Val Loss: 3.8275 Acc: 0.3793 RMSE: 0.2883 | 33.9s
Epoch  6/80 | Train Loss: 4.0594 Acc: 0.3567 | Val Loss: 4.8163 Acc: 0.3809 RMSE: 0.2867 | 32.7s
Epoch  7/80 | Train Loss: 3.1292 Acc: 0.3792 | Val Loss: 3.8511 Acc: 0.4099 RMSE: 0.2790 | 31.3s
Epoch  8/80 | Train Loss: 2.4234 Acc: 0.3817 | Val Loss: 2.5130 Acc: 0.4156 RMSE: 0.2742 | 31.4s
Epoch  9/80 | Train Loss: 2.0905 Acc: 0.3972 | Val Loss: 1.9078 Acc: 0.4404 RMSE: 0.2677 | 31.4s
Epoch 10/80 | Train Loss: 1.9501 Acc: 0.3981 | Val Loss: 1.8050 Acc: 0.4364 RMSE: 0.2682 | 31.4s
Epoch 11/80 | Train Loss: 2.1160 Acc: 0.4025 | Val Loss: 1.8160 Acc: 0.4469 RMSE: 0.2646 | 31.2s
Epoch 12/80 | Train Loss: 2.2666 Acc: 0.4149 | Val Loss: 2.3356 Acc: 0.4526 RMSE: 0.2666 | 31.3s
Epoch 13/80 | Train Loss: 2.06